In [7]:
import json
import os

In [8]:
og_llm_rounds = "./2025-04-26-Multi-Benchmark/llm_outputs"
update_llm_rounds = "./2025-05-04-Multi-Benchmark/llm_outputs_patch"
new_llm_rounds = "./2025-05-04-Multi-Benchmark/llm_outputs"

In [13]:
for round_name in os.listdir(og_llm_rounds):
    og_round_path = os.path.join(og_llm_rounds, round_name)
    update_round_path = os.path.join(update_llm_rounds, round_name)

    for file_name in os.listdir(og_round_path):
        og_file_path = os.path.join(og_round_path, file_name)
        update_file_path = os.path.join(update_round_path, file_name)

        print(f"Checking {og_file_path}")
        with open(og_file_path, "r", encoding="utf-8") as f:
            og_data = json.load(f)

        with open(update_file_path, "r", encoding="utf-8") as f:
            update_data = json.load(f)

        if og_data != update_data:
            print(f"Updating {file_name} in {round_name}")
            og_data.update(update_data)
            new_file_path = os.path.join(new_llm_rounds, round_name, file_name)
            os.makedirs(os.path.dirname(new_file_path), exist_ok=True)
            with open(new_file_path, "w") as f:
                json.dump(og_data, f, indent=4)

Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-claude-3_5-sonnet.json
Updating final_answers-claude-3_5-sonnet.json in round_1
Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-claude-3_7-sonnet.json
Updating final_answers-claude-3_7-sonnet.json in round_1
Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-claude-3_7-sonnet_thinking.json
Updating final_answers-claude-3_7-sonnet_thinking.json in round_1
Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-codestral-2501.json
Updating final_answers-codestral-2501.json in round_1
Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-command-a.json
Updating final_answers-command-a.json in round_1
Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-deepseek-chat-v3-0324.json
Updating final_answers-deepseek-chat-v3-0324.json in round_1
Checking ./2025-04-26-Multi-Benchmark/llm_outputs\round_1\final_answers-deepseek-r1-

In [100]:
import ast

## Find wrong answers by question
eval_json_path = "./2025-05-04-Multi-Benchmark/auto_eval_outputs"
# Dict of arrays of wrong answers
wrong_answers = {}

for model_answers_file in os.listdir(eval_json_path):
    model_name = model_answers_file.replace("auto_eval-", "").replace(".json", "").replace("_", ".")
    file_path = os.path.join(eval_json_path, model_answers_file)
    with open(file_path, "r", encoding="utf-8") as f:
        model_answers_json = json.load(f)

    for _, question in model_answers_json.items():
        if question["score"] < 100:
            # print(f"Wrong answer in {file_name} for question: {question['json_answer']}")
            question_index = question["index"]
            if question_index not in wrong_answers:
                wrong_answers[question_index] = []

            if question["json_answer"] in [None, "None"]:
                continue
            else:
                try:
                    info = ast.literal_eval(question["json_answer"])
                    info["MODEL"] = model_name
                except Exception as e:
                    print("error parsing:", model_name, question["json_answer"])
                    raise e
                q = question["multi_choice_question"]
                answer_str = (
                    q.split("<POSSIBLE ANSWERS>\n")[1].split("\n\n<TASK>")[0].strip().split("\n")
                )
                answer_dict = {
                    answer.split(".")[0]: answer.split(".")[1].strip() for answer in answer_str
                }
                answer_choice = answer_dict[question["json_answer_letter"]]
                answer_choice_str = f"{question['json_answer_letter']}. {answer_choice}"
                info["ANSWER_FULL"] = answer_choice_str
                info["ANSWER_RAW"] = str(answer_choice)

            wrong_answers[question_index].append(info)

# wrong_answers

In [103]:
qs_wrong = sorted(list(wrong_answers.keys()))

question_idx = 12
print("Total wrong answers:", len(wrong_answers[question_idx]))
wrong_answers[question_idx]

json.dumps(wrong_answers)

Total wrong answers: 26


'{"1": [{"ANSWER": "A", "SHORT EXPLANATION": "To find the fastest horse among six horses, you need at least three races. First, race two groups of three horses each. Then race the winners of these two races along with the second-place horse from the race that had the overall winner. This third race will determine the fastest horse overall.", "MODEL": "claude-3.5-sonnet", "ANSWER_FULL": "A. Three", "ANSWER_RAW": "Three"}, {"ANSWER": "D", "SHORT EXPLANATION": "You need at least three races to determine the fastest horse. In each race, you can run up to 6 horses. First race: Run all 6 horses, note top 3. Second race: Run the top 3 from first race to determine 1st, 2nd, and 3rd place. Third race: Run the 4th place from first race with 2nd and 3rd from second race to confirm final rankings. This ensures you find the fastest horse with certainty.", "MODEL": "claude-3.5-sonnet", "ANSWER_FULL": "D. Three", "ANSWER_RAW": "Three"}, {"ANSWER": "B", "SHORT EXPLANATION": "With six horses, you need 